# IndieFake (Indian-accent) SSL training pipeline -- Colab

Picks up the `ai_voice_detector` pipeline at step 4 (SSL feature extraction), which was too slow on an 8GB CPU-only machine. Uses Colab's GPU (`Runtime > Change runtime type > T4 GPU`) instead.

**Before running**: put the 4 IndieFake zip parts (`drive-download-...-00{1..4}.zip`) somewhere in your Google Drive and set `INDIEFAKE_ZIP_DIR` below to that folder.

Steps: mount Drive -> clone repo -> re-fetch ASVspoof + real-world audio (public/deterministic sources, not stored in git) -> run the Indian dataset pipeline (organize -> verify -> degrade -> manifests -> extract embeddings on GPU) -> train -> evaluate (5-quadrant + leave-one-generator-out) -> push results back.

In [7]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [8]:
# EDIT THIS to wherever the 4 IndieFake zip parts live in your Drive
INDIEFAKE_ZIP_DIR = "/content/drive/MyDrive/IndieFake_Datasets/IndieFake_Dataset"

In [9]:
REPO_URL = "https://github.com/Jeevan-Cyber-Sai/sih.git"
!git clone $REPO_URL /content/sih
%cd /content/sih/ai_voice_detector

Cloning into '/content/sih'...
remote: Enumerating objects: 6718, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 6718 (delta 13), reused 24 (delta 10), pack-reused 6685 (from 1)
Receiving objects: 100% (6718/6718), 783.04 MiB | 31.02 MiB/s, done.
Resolving deltas: 100% (63/63), done.
Updating files: 100% (11681/11681), done.
/content/sih/ai_voice_detector


In [10]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 52.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 1.16.1 which is incompatible.


In [11]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- set Runtime > Change runtime type > T4 GPU')

CUDA available: True
device: Tesla T4


## Re-fetch ASVspoof + real-world audio

`data/`, `data_realworld/` audio isn't in git (gitignored, large binary data) -- only their SSL embeddings are (`cache/ssl_embeddings/`, portable across machines since the cache key is now a relative path, not an absolute one). These scripts rebuild the audio deterministically (same seeds, same public sources) so file *names* line up with what's already cached; `LA_D_*`/WhatsApp/test-dir files come from the same seeded/tracked sources so they're byte-identical, while gTTS/LibriSpeech content could differ by a negligible amount from re-synthesis -- acceptable since only filenames need to match for cache lookups, and any recompute needed for a mismatch is cheap on GPU anyway.

In [12]:
CACHE_ZIP_PATH = "/content/drive/MyDrive/IndieFake_Datasets/IndieFake_Dataset/asvspoof_realworld_cache.zip"  # edit if you put it elsewhere
!unzip -oq "$CACHE_ZIP_PATH" -d /content/sih/ai_voice_detector
print('restored data/, data_realworld/, cache/ssl_embeddings/ from Drive')


restored data/, data_realworld/, cache/ssl_embeddings/ from Drive


In [13]:
!python scripts/build_realworld_dataset.py

[08:37:31] converting WhatsApp voice notes...
[08:37:31] WhatsApp: 3 files converted
[08:37:31] downloading LibriSpeech dev-clean (https://www.openslr.org/resources/12/dev-clean.tar.gz)...
[08:37:55] download complete: 337926286 bytes
[08:37:55] scanning tarball for .flac entries...
[08:37:56] found 2703 flac utterances in archive
[08:37:56] LibriSpeech extraction done: 220/220 converted
[08:37:56] DONE. data_realworld/real/ now has 222 wav files (whatsapp=3, librispeech=220)


In [14]:
!pip install -q gtts requests

In [15]:
!python scripts/build_realworld_fake_dataset.py

[08:38:48] holdout exclude set: 413 files
[08:38:48] building modern TTS fake samples (gTTS)...
[08:38:57] loaded 200 candidate sentences from LibriSpeech transcripts
[08:39:14] modern TTS fake: 200/200 converted
[08:39:14] building degraded copies of ASVspoof fake samples...
[08:39:18]   degraded 50/200
[08:39:22]   degraded 100/200
[08:39:26]   degraded 150/200
[08:39:31]   degraded 200/200
[08:39:31] degraded ASVspoof copies: 200/200 converted
[08:39:31] DONE. data_realworld/fake/ now has 537 wav files (tts=200, degraded_asvspoof=200)


In [16]:
!cd /content/sih && git pull


Already up to date.


In [17]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f.lower().endswith('.zip'):
            print(os.path.join(root, f))

/content/drive/MyDrive/IndieFake_Datasets/IndieFake_Dataset/asvspoof_data.zip
/content/drive/MyDrive/IndieFake_Datasets/IndieFake_Dataset/drive-download-20260903T071406Z-1-004.zip
/content/drive/MyDrive/IndieFake_Datasets/IndieFake_Dataset/drive-download-20260903T071406Z-1-001.zip
/content/drive/MyDrive/IndieFake_Datasets/IndieFake_Dataset/drive-download-20260903T071406Z-1-002.zip
/content/drive/MyDrive/IndieFake_Datasets/IndieFake_Dataset/drive-download-20260903T071406Z-1-003.zip
/content/drive/MyDrive/IndieFake_Datasets/IndieFake_Dataset/asvspoof_realworld_cache.zip


## Indian dataset pipeline (steps 1-3, replayed here since they're cheap)

In [18]:
import os
os.environ['INDIEFAKE_ZIP_DIR'] = INDIEFAKE_ZIP_DIR
!python scripts/organize_indian_dataset.py

found 6 zip part(s) in /content/drive/MyDrive/IndieFake_Datasets/IndieFake_Dataset: ['asvspoof_data.zip', 'asvspoof_realworld_cache.zip', 'drive-download-20260903T071406Z-1-001.zip', 'drive-download-20260903T071406Z-1-002.zip', 'drive-download-20260903T071406Z-1-003.zip', 'drive-download-20260903T071406Z-1-004.zip']
extracting asvspoof_data.zip ...
extracting asvspoof_realworld_cache.zip ...
extracting drive-download-20260903T071406Z-1-001.zip ...
extracting drive-download-20260903T071406Z-1-002.zip ...
extracting drive-download-20260903T071406Z-1-003.zip ...
extracting drive-download-20260903T071406Z-1-004.zip ...

real (Bonafides) -> /content/sih/ai_voice_detector/data_indian/real: 8189 files
fake (Deepfakes) -> /content/sih/ai_voice_detector/data_indian/fake: 11426 files
skipped (unrecognized path/label): 3437


In [19]:
!python verify_indian_dataset.py

/content/sih/ai_voice_detector/verify_indian_dataset.py:41: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/content/sih/ai_voice_detector/verify_indian_dataset.py:41: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/content/sih/ai_voice_detector/verify_indian_dataset.py:41: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr

In [20]:
!python scripts/degrade_indian_dataset.py

total files to process: 19514 (real=8172, fake=11342)
  processed 1000/19514 (116s elapsed)
  processed 2000/19514 (234s elapsed)
  processed 3000/19514 (340s elapsed)
  processed 4000/19514 (460s elapsed)
  processed 5000/19514 (573s elapsed)
  processed 6000/19514 (689s elapsed)
  processed 7000/19514 (806s elapsed)
  processed 8000/19514 (920s elapsed)
  processed 9000/19514 (1030s elapsed)
  processed 10000/19514 (1140s elapsed)
  processed 11000/19514 (1255s elapsed)
  processed 12000/19514 (1359s elapsed)
  processed 13000/19514 (1474s elapsed)
  processed 14000/19514 (1583s elapsed)
  processed 15000/19514 (1690s elapsed)
  processed 16000/19514 (1796s elapsed)
  processed 17000/19514 (1904s elapsed)
  processed 18000/19514 (2004s elapsed)
  processed 19000/19514 (2107s elapsed)

DONE in 2168.8s: ok=19514 skipped(existing)=0 errors=0
data_indian_augmented/real/: 8172 files
data_indian_augmented/fake/: 11342 files

clean (left untouched): 5861 (30.0%)
degraded: 13653 (70.0%)

deg

In [21]:
# Idempotent -- these will report 0 new entries since the manifests are
# already committed to git and cloned above. Harmless to (re-)run.
!python scripts/build_holdout_manifest_indian.py
!python scripts/build_generator_manifest_indian.py

indian_real += 0
indian_fake += 0
total manifest size: 413
indiefake += 0
total generator manifest size: 24629


## Step 4: extract SSL embeddings on GPU

`features_ssl.py` now auto-detects CUDA and moves the model/inputs there. `MAX_WORKERS=1` in the script is intentional even here -- one process already saturates a single GPU; more workers would just fight over it via separate CUDA contexts.

In [22]:
!python scripts/extract_indian_ssl_features.py

holdout exclude: 413 exact paths, 60 indian basenames (also excluded from data_indian_augmented/)
total files to embed: 38908 (excluded: 120)
preprocessor_config.json: 100% 212/212 [00:00<00:00, 1.03MB/s]
config.json: 100% 1.57k/1.57k [00:00<00:00, 4.85MB/s]
pytorch_model.bin: 100% 1.27G/1.27G [00:11<00:00, 107MB/s]
Loading weights: 100% 134/134 [00:00<00:00, 21093.52it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                                                                     | Status     |  | 
------------------------------------------------------------------------+------------+--+-
wav2vec2.encoder.layers.{6...23}.final_layer_norm.weight                | UNEXPECTED |  | 
wav2vec2.encoder.layers.{6...23}.feed_forward.intermediate_dense.bias   | UNEXPECTED |  | 
wav2vec2.encoder.layers.{6...23}.final_layer_norm.bias                  | UNEXPECTED |  | 
wav2vec2.encoder.layers.{6...23}.attention.q_proj.weight                | UNEXPECTED |  | 
wa

## Step 5: train the combined classifier

In [24]:
!cd /content/sih && git pull


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.05 KiB | 537.00 KiB/s, done.
From https://github.com/Jeevan-Cyber-Sai/sih
   9a9a788..31bcfa1  main       -> origin/main
Updating 9a9a788..31bcfa1
Fast-forward
 ai_voice_detector/train_ssl.py | 13 +++++++++++--
 1 file changed, 11 insertions(+), 2 deletions(-)


In [25]:
!python train_ssl.py --indian

using facebook/wav2vec2-xls-r-300m (matches features_indian.npy)

extracting SSL embeddings (facebook/wav2vec2-xls-r-300m, layer 6) for up to 2824 files (per-file cache in /content/sih/ai_voice_detector/cache/ssl_embeddings)...
processed 100 files... (cache hits so far: 100)
processed 200 files... (cache hits so far: 200)
processed 300 files... (cache hits so far: 300)
processed 400 files... (cache hits so far: 400)
processed 500 files... (cache hits so far: 500)
processed 600 files... (cache hits so far: 600)
processed 700 files... (cache hits so far: 700)
processed 800 files... (cache hits so far: 800)
processed 900 files... (cache hits so far: 900)
processed 1000 files... (cache hits so far: 1000)
processed 1100 files... (cache hits so far: 1100)
processed 1200 files... (cache hits so far: 1200)
processed 1300 files... (cache hits so far: 1300)
processed 1400 files... (cache hits so far: 1400)
processed 1500 files... (cache hits so far: 1500)
processed 1600 files... (cache hits so f

## Step 6: six-bucket evaluation (clean / real-world / Indian x real / fake)

In [27]:
!python scripts/five_quadrant_eval.py

clean_real: 30 held-out files
clean_fake: 30 held-out files
realworld_real: 20 held-out files
realworld_fake: 273 held-out files
indian_real: 30 held-out files
indian_fake: 30 held-out files

skipping SSL v2 (pre-Indian, ASVspoof+realworld): /content/sih/ai_voice_detector/models/voice_classifier_ssl_v2.joblib not found
evaluating SSL indian (ASVspoof+realworld+IndieFake)...
Loading weights: 100% 134/134 [00:00<00:00, 12256.56it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                                                                     | Status     |  | 
------------------------------------------------------------------------+------------+--+-
wav2vec2.encoder.layers.{6...23}.feed_forward.output_dense.weight       | UNEXPECTED |  | 
wav2vec2.encoder.layers.{6...23}.final_layer_norm.bias                  | UNEXPECTED |  | 
wav2vec2.encoder.layers.{6...23}.attention.v_proj.weight                | UNEXPECTED |  | 
wav2vec2.encoder.layers.{6...23}.a

## Step 7: leave-one-generator-out, including the new 'indiefake' generator

In [28]:
!python scripts/leave_one_generator_out_eval.py

real pool (excluding four-quadrant holdout): 872 files
  generator 'indiefake': 22684 fake samples
  generator 'asvspoof': 908 fake samples
  generator 'elevenlabs': 336 fake samples
  generator 'respeecher': 265 fake samples
  generator 'gtts': 199 fake samples
  generator 'sapi': 60 fake samples
  generator 'piper': 60 fake samples
  generator 'edgetts': 60 fake samples
  generator 'knnvc': 57 fake samples

matched real test set (fixed, reused across all folds): 50 files
real training pool: 822 files

pre-embedding real train/test pools...
done.

=== held out: asvspoof (908 test samples) ===
Loading weights: 100% 422/422 [00:00<00:00, 16651.74it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias           

## Bring results back

`models/*.joblib` and `features_indian.npy` are gitignored (large/regenerable) -- copy them to Drive to download. The new Indian embeddings in `cache/ssl_embeddings/` ARE meant to be committed (same convention as the existing ASVspoof/real-world cache) so any machine that later `git pull`s gets them for free -- this needs a GitHub token with push access to this repo, entered interactively below (never hardcode it in the notebook).

In [29]:
!mkdir -p /content/drive/MyDrive/sih_indian_results
!cp models/voice_classifier_ssl_indian.joblib models/scaler_ssl_indian.joblib features_indian.npy /content/drive/MyDrive/sih_indian_results/
print('copied to Drive: sih_indian_results/')

copied to Drive: sih_indian_results/


In [30]:
import getpass
token = getpass.getpass('GitHub personal access token (repo scope, push rights): ')
!git config user.email "jeevanarhack@gmail.com"
!git config user.name "Jeevan Sai V"
!git add cache/ssl_embeddings data_realworld/holdout_manifest.json data_realworld/generator_manifest.json
!git commit -m "Add Indian-accent SSL embeddings from Colab run"
!git push https://{token}@github.com/Jeevan-Cyber-Sai/sih.git HEAD:main

Streaming output truncated to the last 5000 lines.
 create mode 100644 ai_voice_detector/cache/ssl_embeddings/de8f60332f942cd64f418fbe5c63fed11fc49563.npy
 create mode 100644 ai_voice_detector/cache/ssl_embeddings/de90db9ff69d5bcc17e44c86e66b288fc3a24c74.npy
 create mode 100644 ai_voice_detector/cache/ssl_embeddings/de9344040445bb3a88604ade2b3bb8a596f4d182.npy
 create mode 100644 ai_voice_detector/cache/ssl_embeddings/de9623d3e6769e81b63778cd28f0e5a751474087.npy
 create mode 100644 ai_voice_detector/cache/ssl_embeddings/de975f1ff4441aa465b60c85dc7910eff7b48e55.npy
 create mode 100644 ai_voice_detector/cache/ssl_embeddings/de998bab65a61f896b41b31386efd18ca1823b00.npy
 create mode 100644 ai_voice_detector/cache/ssl_embeddings/de9a2098a16cc342f6ca5ce9fcc30a9854c9b7de.npy
 create mode 100644 ai_voice_detector/cache/ssl_embeddings/de9c3a91ed7dbd8bc83d23ed341391d50323b44e.npy
 create mode 100644 ai_voice_detector/cache/ssl_embeddings/dea0da5e31dfbbad9a8a0a6d3d25f248bb53335e.npy
 create mode 

In [31]:
!git pull
!git push https://{token}@github.com/Jeevan-Cyber-Sai/sih.git HEAD:main

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.48 KiB | 1.48 MiB/s, done.
From https://github.com/Jeevan-Cyber-Sai/sih
   31bcfa16..f1c32faa  main       -> origin/main
hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint: 
hint:   git config pull.rebase false  # merge (the default strategy)
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint: 
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-only on the command line to override the configured default per
hint: invocation.
fatal: Need to specify how t

In [35]:
!git commit --no-edit
!git push https://{token}@github.com/Jeevan-Cyber-Sai/sih.git HEAD:main



On branch main
Your branch is ahead of 'origin/main' by 2 commits.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   data/download_manifest.csv
	modified:   scripts/_gtts_tmp/tts_0098.mp3

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	ssl_embeddings/

no changes added to commit (use "git add" and/or "git commit -a")
Everything up-to-date
